<a href="https://colab.research.google.com/github/Aksinhaa/ColabFold/blob/main/TGW_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
# Install Miniconda
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh -b -p /usr/local/miniconda

# Initialize conda
source /usr/local/miniconda/etc/profile.d/conda.sh

# Accept Terms of Service
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Add channels
conda config --add channels defaults
conda config --add channels bioconda
conda config --add channels conda-forge

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda create -y -n ngs_env angsd plink qualimap

In [ ]:
%%bash

mkdir -p /content/pca_plink
cd /content/pca_plink

In [ ]:
%%bash

cd /content/pca_plink

wget -nc https://zenodo.org/records/19914378/files/all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.bed
wget -nc https://zenodo.org/records/19914378/files/all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.bim
wget -nc https://zenodo.org/records/19914378/files/all_pigeons_newIDs-onlyTrans-mf01-noOut-geno.fam

In [ ]:
%%bash

cd /content/pca_plink

cat << 'EOF' > pop_names.txt
IID Population
Niger Africa
Hoggar1 Africa
Senegal-P Africa
Hoggar2 Africa
Hoggar3 Africa
Senegal Africa
fantail Domestic
Starling Domestic
Scandaroon Domestic
Carneau Domestic
Jacobin Domestic
Lahore Domestic
Lebanon MiddleEast
Cumulet Domestic
Runt Domestic
Englsh_trumpt Domestic
Racing Domestic
Oriental Domestic
Iran1 MiddleEast
Algeria1 Africa
Kashmir2 Asia
Hebrides1 Europe
Mykonos Europe
Libya1 Africa
Egypt1 Africa
Gebeit_Sudan Africa
Athlit MiddleEast
Jerico MiddleEast
Iraq3 MiddleEast
Tunisia Africa
Libya2 Africa
Algeria2 Africa
Ireland Europe
Orkney Europe
Italy2 Europe
Ghana Africa
Mali Africa
Kashmir1 Asia
Gebeit_Sudan-T Africa
Sudan Africa
Darfur_Sudan1 Africa
Mali-T Africa
Darfur_Sudan2 Africa
Saudi MiddleEast
Egypt2 Africa
Oman MiddleEast
EOF

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/pca_plink

plink \
  --bfile all_pigeons_newIDs-onlyTrans-mf01-noOut-geno \
  --indep-pairwise 50 10 0.4 \
  --out LD_prune \
  --allow-extra-chr

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env

cd /content/pca_plink

plink \
  --bfile all_pigeons_newIDs-onlyTrans-mf01-noOut-geno \
  --extract LD_prune.prune.in \
  --make-bed \
  --out geno_LDpruned \
  --allow-extra-chr

In [ ]:
%%bash
source /usr/local/miniconda/etc/profile.d/conda.sh
conda activate ngs_env
cd /content/pca_plink

plink \
  --bfile geno_LDpruned \
  --pca 10 \
  --out pca \
  --allow-extra-chr

In [ ]:
%%bash

cat << 'EOF' > /content/pca_plink/pca_plot.R

library(ggplot2)
library(ggrepel)

# Load PCA
pca <- read.table("pca.eigenvec", header = FALSE)
colnames(pca) <- c("FID", "IID", "PC1", "PC2")

# Variance
eig <- scan("pca.eigenval")
var_exp <- eig / sum(eig) * 100

# Load population info
pop <- read.table("pop_names.txt", header = TRUE)

# Merge
df <- merge(pca, pop, by = "IID")

# Plot
p <- ggplot(df, aes(PC1, PC2, colour = Population)) +
  geom_point(size = 3) +
  geom_text_repel(aes(label = IID)) +
  theme_minimal() +
  xlab(paste0("PC1 (", round(var_exp[1], 2), "%)")) +
  ylab(paste0("PC2 (", round(var_exp[2], 2), "%)"))

ggsave("pca_plot.png", p, width = 6, height = 5)

print(p)

EOF

In [ ]:
%%bash
cd /content/pca_plink

echo "PCA file:"
head pca.eigenvec

echo "POP file:"
head pop_names.txt

In [ ]:
%%bash
Rscript -e 'install.packages("ggrepel", repos="http://cran.us.r-project.org")'
cd /content/pca_plink
Rscript pca_plot.R

In [ ]:
from IPython.display import Image
Image("/content/pca_plink/pca_plot.png")